# Vector算子计算模型与典型场景

## 概述

本节介绍Vector（矢量）算子的计算模型与典型应用场景。第2章我们学习了pyasc核函数开发流程，其中的多核并行遵循**SPMD**模型——每个AI Core执行同一份核函数、通过`asc.get_block_idx()`处理不同数据分片。本节在此基础上深入单个AI Core**内部**：昇腾NPU的并行体系是「外层多核SPMD + 内层单核SIMD」的双层嵌套结构，Vector算子的计算正是依托单核内的**SIMD**并行机制。理解SIMD计算模型与Vector单元的硬件定位，是掌握Vector算子工程化开发的第一步。

### 学习目标

完成本节后，开发者应能够：

1. 理解SPMD与SIMD的嵌套关系，明确本节聚焦的是单核内SIMD并行；
2. 理解SIMD（单指令多数据）计算模型的核心概念与特征；
3. 认识AI Core的计算单元构成，明确Vector算子由向量处理单元承载；
4. 认识element-wise（逐元素）典型场景及对应的pyasc矢量计算接口。

In [ ]:
# 环境初始化
import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("环境初始化完成")

---
# 1. 从SPMD到SIMD：单核内的并行

昇腾NPU的异构并行体系采用**外层多核SPMD并行 + 内层单核SIMD细粒度并行**的双层嵌套结构：

- **多核集群层面遵循SPMD（Single Program Multiple Data）**：每个AI Core抽象为一个Block，通过`block_idx`索引，每个Block执行同一份核函数、处理不同的数据分片，实现多核负载均衡。这正是第2章已经介绍的多核并行方式。
- **单核计算层面依托SIMD**：单个AI Core内部通过SIMD机制实现细粒度并行，一条指令批量完成多组同质数据的并行运算，充分挖掘单核算力。

也就是说，第2章讲的SPMD决定了「数据如何分配到多个核」，而本节要讲的SIMD决定了「单个核内部如何一次并行处理一批数据」。二者是嵌套递进关系，共同构成Vector算子的完整并行模型。

# 2. SIMD计算模型

**SIMD（Single Instruction Multiple Data，单指令多数据）** 是一种数据并行模型，核心逻辑是：一条指令在同一个时钟周期内，对多个数据元素执行完全相同的操作，实现数据的批量并行处理。

SIMD具有三个核心特征：

- **单指令驱动**：所有并行计算单元同步执行同一条指令，操作完全一致；
- **数据同构**：要求参与计算的数据类型统一、长度相同，确保指令可批量处理；
- **同步执行**：所有数据的操作在同一个指令周期内完成，执行节奏统一。

以矢量加法为例，一条`asc.add`指令即可完成一段连续数据的逐元素相加：

```text
输入x:  [x0, x1, x2, x3, ...]
输入y:  [y0, y1, y2, y3, ...]
                |  asc.add（一条指令批量处理一批元素）
输出z:  [x0+y0, x1+y1, x2+y2, x3+y3, ...]
```

正因为要求「数据同构、操作规整」，SIMD最适合处理**规则、连续、逐元素**的计算，这也决定了Vector算子的典型计算形态。

---
# 3. AI Core计算单元

SIMD并行模型在昇腾AI处理器中有明确的硬件对应。Device端的核心计算载体是**AI Core**，单枚NPU芯片通常集成多个AI Core。每个AI Core内部分工明确，核心组件包括：

- **标量处理单元**：AI Core的控制中枢，负责地址计算、指令调度与发射，支撑分支、循环等控制流；
- **向量处理单元（Vector）**：遵循SIMD并行计算逻辑，专职执行各类向量指令，适配元素级计算、逻辑运算等场景；
- **矩阵运算单元（Cube）**：面向矩阵乘加、卷积等算力密集场景做深度优化；
- **本地存储**：AI Core片内高速存储，缓存计算数据以规避全局内存的高延迟访问。其中Vector计算单元配套统一缓存**UB（Unified Buffer）**。

根据核心计算逻辑对硬件单元的依赖差异，AI Core算子可划分为三类标准类型：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr><th align="left">算子类型</th><th align="left">承载单元</th><th align="left">计算特征</th></tr>
  </thead>
  <tbody>
    <tr><td><b>矢量类算子</b></td><td>Vector计算单元</td><td>以元素级、逻辑类、数据重组类计算为主，访存密集、灵活度高</td></tr>
    <tr><td>矩阵类算子</td><td>Cube计算单元</td><td>以大维度矩阵乘、卷积等规整算力密集计算为主</td></tr>
    <tr><td>融合类算子</td><td>Cube + Vector协同</td><td>兼顾矩阵高密度算力与向量灵活逻辑处理</td></tr>
  </tbody>
</table>

**本章聚焦的Vector算子，就是核心计算逻辑完全由Vector计算单元承载的矢量类算子。** 它基于SIMD并行、在UB上完成计算，是昇腾NPU上最基础、使用最广泛的一类算子。

---
# 4. element-wise典型计算场景与pyasc接口

Vector算子最典型的场景是**element-wise（逐元素）计算**：输出张量的每个元素只依赖输入张量对应位置的元素。SIMD适配的典型计算任务中，逐元素数学函数（向量加减乘除、指数/对数等）正是这一类。

pyasc提供了与Ascend C一一对应的矢量计算接口。按操作数形态，逐元素接口可分为三类，常用接口如下：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr><th align="left">类别</th><th align="left">pyasc接口</th><th align="left">计算含义</th></tr>
  </thead>
  <tbody>
    <tr><td rowspan="6">双目<br>（两个输入）</td><td><code>asc.add</code></td><td>逐元素相加 z = x + y</td></tr>
    <tr><td><code>asc.sub</code></td><td>逐元素相减 z = x - y</td></tr>
    <tr><td><code>asc.mul</code></td><td>逐元素相乘 z = x * y</td></tr>
    <tr><td><code>asc.div</code></td><td>逐元素相除 z = x / y</td></tr>
    <tr><td><code>asc.max</code></td><td>逐元素求最大值</td></tr>
    <tr><td><code>asc.min</code></td><td>逐元素求最小值</td></tr>
    <tr><td rowspan="5">单目<br>（单个输入）</td><td><code>asc.relu</code></td><td>激活 z = max(x, 0)</td></tr>
    <tr><td><code>asc.abs</code></td><td>绝对值 z = |x|</td></tr>
    <tr><td><code>asc.exp</code></td><td>自然指数 z = exp(x)</td></tr>
    <tr><td><code>asc.ln</code></td><td>自然对数 z = ln(x)</td></tr>
    <tr><td><code>asc.sqrt</code></td><td>开方 z = sqrt(x)</td></tr>
    <tr><td rowspan="3">标量<br>（输入 + 标量）</td><td><code>asc.adds</code></td><td>加标量 z = x + scalar</td></tr>
    <tr><td><code>asc.muls</code></td><td>乘标量 z = x * scalar</td></tr>
    <tr><td><code>asc.leaky_relu</code></td><td>带泄漏的激活</td></tr>
  </tbody>
</table>

这些接口的调用形式高度统一，以最常用的简化形式为例：

```python
asc.add(z_local, x_local, y_local, count)      # 双目：目标、源0、源1、元素个数
asc.relu(z_local, x_local, count)              # 单目：目标、源、元素个数
asc.muls(z_local, x_local, scalar, count)      # 标量：目标、源、标量、元素个数
```

其中`count`为本次计算处理的元素个数，操作数均为位于片上内存（UB）的`LocalTensor`。可以看到，无论算子是加、乘还是ReLU，调用范式几乎一致——这正是Vector算子可以工程化、模板化开发的基础。

> 逐元素接口对应的典型场景：逐元素算术（add/sub/mul/div）、激活（relu/leaky_relu）、数值函数（exp/ln/sqrt）等。pyasc还提供归约类等更复杂的矢量接口，将在后续内容中涉及。

---
# 5. 从计算模型到工程化开发

Vector单元只能对片上内存（UB）中的数据进行计算，而算子输入输出数据初始位于Global Memory，因此第2章介绍的**CopyIn→Compute→CopyOut三段式数据流**同样适用于所有Vector算子。

结合本节的接口全景，可以得到一个贯穿后续开发的关键规律：**不同Vector算子的差异只集中在Compute阶段的计算接口**——CopyIn与CopyOut逻辑完全一致。把三段式流程固定为模板、只替换Compute阶段的矢量接口，即可快速实现一族Vector算子。这正是后续工程化、模板化开发的核心思路。

# 6. 小结

本节介绍了Vector算子的计算模型与典型场景：

- **SPMD与SIMD的嵌套关系**：外层多核SPMD分配数据分片，内层单核SIMD批量并行处理，本节聚焦单核内的SIMD；
- **SIMD计算模型**：一条指令对多个同构数据执行相同操作，具备单指令驱动、数据同构、同步执行三个特征；
- **Vector算子定位**：由AI Core的向量处理单元承载、基于UB计算的矢量类算子；
- **element-wise典型场景**：双目（add/sub/mul/div/max/min）、单目（relu/abs/exp/ln/sqrt）、标量（adds/muls/leaky_relu）等逐元素接口，调用范式统一。

---
## 课后练习

**选择题：**

1. 关于昇腾NPU的并行模型，以下说法正确的是？
   - A. SPMD与SIMD是互相独立、不相关的两种模型
   - B. 外层多核遵循SPMD、内层单核依托SIMD，二者是嵌套关系
   - C. Vector算子只使用SPMD，不使用SIMD
   - D. SIMD决定数据如何分配到多个核

2. SIMD计算模型的核心特征**不**包括以下哪一项？
   - A. 单指令驱动，所有计算单元执行同一条指令
   - B. 数据同构，参与计算的数据类型统一、长度相同
   - C. 各数据元素可按不同分支独立执行
   - D. 同步执行，操作在同一指令周期内完成

3. Vector算子的核心计算逻辑由AI Core的哪个单元承载？
   - A. 标量处理单元
   - B. 向量处理单元（Vector）
   - C. 矩阵运算单元（Cube）
   - D. 本地存储

4. 以下哪个pyasc接口属于**标量**类（输入 + 标量）计算接口？
   - A. `asc.add`
   - B. `asc.relu`
   - C. `asc.muls`
   - D. `asc.max`

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.02_answer.txt